# benchmarking_atari_50m_realv1

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import importlib.util
from pathlib import Path

import matplotlib.pyplot as plt

from analysis.curves import return_curve, subsample
from analysis.stats import mean_ci
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"
REFERENCE_CONFIG = Path("../atari_50m_deterministic_realv1/config.py")

In [ ]:
# Load Agent0's stored runs per game, and the DQN runs of
# atari_50m_deterministic_realv1 as the reference.
AGENT0 = "Agent0"
DQN = "DQN"
component = EXPERIMENT.component("agent0_atari")
GAMES = component.sweep["ENV_HYPERS.GAME"]
FRAMESKIP = component.config.ENV_HYPERS.FRAMESKIP
TOTAL_FRAMES = component.config.AGENT_HYPERS.TOTAL_TIMESTEPS * FRAMESKIP

spec = importlib.util.spec_from_file_location("reference_config", REFERENCE_CONFIG)
reference = importlib.util.module_from_spec(spec)
spec.loader.exec_module(reference)
STORES = {
    AGENT0: (EXPERIMENT, "agent0_atari"),
    DQN: (reference.EXPERIMENT, "dqn_atari"),
}

results = {}
for agent, (experiment, name) in STORES.items():
    for game in GAMES:
        runs = load_results(experiment, name, where={"ENV_HYPERS.GAME": game})
        if runs is not None:
            results[agent, game] = runs

In [ ]:
# Compute each agent's mean return per game with a 95% bootstrap CI.
curves = {}
for (agent, game), runs in results.items():
    timesteps, returns = subsample(return_curve(runs.reward, runs.done))
    curves[agent, game] = (timesteps * FRAMESKIP, mean_ci(returns))

In [ ]:
# Style for the all-games grid.
LINES = {
    DQN: {"color": "tab:blue", "linewidth": 1.75, "label": "DQN"},
    AGENT0: {"color": "tab:red", "linewidth": 1.75, "label": "Agent0"},
}
BAND_ALPHA = 0.2
GRID_STYLE = {
    "figure.figsize": (16, 9.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 20,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 20,
    "legend.frameon": False,
    "figure.labelsize": 20,
}
GRID_SHAPE = (3, 4)
PANELS = {  # (column, row) of each game's panel
    "battle_zone": (0, 0),
    "double_dunk": (0, 1),
    "name_this_game": (0, 2),
    "phoenix": (1, 0),
    "qbert": (1, 1),
    "pong": (1, 2),
    "breakout": (2, 0),
    "seaquest": (2, 1),
    "centipede": (2, 2),
    "ms_pacman": (3, 0),
    "beam_rider": (3, 1),
    "space_invaders": (3, 2),
}
PANEL_ASPECT = 3 / 4
GRID_XTICKS = [0, TOTAL_FRAMES]
GRID_XTICK_LABELS = ["0", f"{TOTAL_FRAMES / 1_000_000:.0f}M"]
GRID_LAYOUT = {"rect": (0.03, 0.06, 1, 1), "w_pad": 0.2, "h_pad": 0.2}
GRID_XLABEL = {"t": "Frames", "y": 0.055, "va": "top"}
GRID_YLABEL = {"t": "Return"}
GRID_LEGEND = {"loc": "upper center", "bbox_to_anchor": (0.5, 0.035), "ncol": 2}
COMPARISONS = {"grid_dopamine_ref": "Dopamine Ref", "grid_agent0": "Agent0"}


def title(game):
    return game.replace("_", " ").title()


def draw(ax, agent, game):
    frames, (mean, low, high) = curves[agent, game]
    line = LINES[agent]
    ax.fill_between(frames, low, high, color=line["color"], alpha=BAND_ALPHA)
    ax.plot(frames, mean, **line)

In [ ]:
# Plot every game in one grid: Agent0 against the DQN reference.
with plt.rc_context(GRID_STYLE):
    fig, axes = plt.subplots(*GRID_SHAPE)
    legend = {}
    for game, (column, row) in PANELS.items():
        ax = axes[row, column]
        ax.set_box_aspect(PANEL_ASPECT)
        drawn = [agent for agent in LINES if (agent, game) in curves]
        if not drawn:
            ax.axis("off")
            continue
        for agent in drawn:
            draw(ax, agent, game)
        ax.set_title(title(game))
        ax.set_xticks(GRID_XTICKS, GRID_XTICK_LABELS)
        handles, labels = ax.get_legend_handles_labels()
        legend.update(zip(labels, handles, strict=True))
    fig.tight_layout(**GRID_LAYOUT)
    fig.supxlabel(**GRID_XLABEL)
    fig.supylabel(**GRID_YLABEL)
    fig.legend(legend.values(), legend.keys(), **GRID_LEGEND)
figures = {"grid": fig}
plt.show()

In [ ]:
# Save every plotted figure as PDF and PNG in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    for ext in ("pdf", "png"):
        fig.savefig(PLOTS_DIR / f"{name}.{ext}", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")